In [0]:
# Install Great Expectations
%pip install great_expectations

print("Great Expectations installed successfully")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Great Expectations installed successfully


In [0]:
# Imports
import great_expectations as gx
from great_expectations.dataset import SparkDFDataset

print("Imports successful")

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-7576825963721843>, line 3
      1 # Imports
      2 import great_expectations as gx
----> 3 from great_expectations.dataset import SparkDFDataset
      5 print("Imports successful")

ModuleNotFoundError: No module named 'great_expectations.dataset'

In [0]:
# Imports
import great_expectations as gx
from great_expectations.core import ExpectationSuite

print(f"Great Expectations version: {gx.__version__}")
print("Imports successful")

Great Expectations version: 1.23.0
Imports successful


In [0]:
# Read Bronze claims for validation
bronze_claims = spark.sql("SELECT * FROM bronze.claims")

print(f"Total records to validate: {bronze_claims.count()}")
bronze_claims.show()

Total records to validate: 9
+--------+----------+-----------+------------+------------+-------------------+--------------------+--------+
|claim_id|patient_id|provider_id|claim_status|claim_amount|         updated_at|        _ingested_at| _source|
+--------+----------+-----------+------------+------------+-------------------+--------------------+--------+
|  CLM001|    PAT101|      PRV01|    APPROVED|      1500.0|2024-01-15 10:30:00|2026-09-10 01:19:...|REST_API|
|  CLM002|    PAT102|      PRV02|     PENDING|       800.0|2024-01-15 11:00:00|2026-09-10 01:19:...|REST_API|
|  CLM003|    PAT103|      PRV03|      DENIED|      2200.0|2024-01-15 12:00:00|2026-09-10 01:19:...|REST_API|
|  CLM004|    PAT104|      PRV01|    APPROVED|      3100.0|2024-01-15 13:00:00|2026-09-10 01:19:...|REST_API|
|  CLM005|    PAT105|      PRV02|     PENDING|       950.0|2024-01-15 14:00:00|2026-09-10 01:19:...|REST_API|
|  CLM006|    PAT106|      PRV03|    APPROVED|      4500.0|2024-01-16 09:00:00|2026-09-10 0

In [0]:
# Define data quality rules for claims
def validate_claims(df):
    
    # Convert Spark DataFrame to Pandas
    pandas_df = df.toPandas()
    
    # Run validation rules manually
    results = {}
    
    # Rule 1 - claim_id should never be null
    results["claim_id not null"] = pandas_df["claim_id"].isnull().sum() == 0
    
    # Rule 2 - claim_amount should always be positive
    results["claim_amount positive"] = (pandas_df["claim_amount"] > 0).all()
    
    # Rule 3 - claim_status must be valid values only
    valid_statuses = ["PENDING", "APPROVED", "DENIED", "PAID"]
    results["claim_status valid values"] = pandas_df["claim_status"].isin(valid_statuses).all()
    
    # Rule 4 - patient_id should never be null
    results["patient_id not null"] = pandas_df["patient_id"].isnull().sum() == 0
    
    # Rule 5 - provider_id should never be null
    results["provider_id not null"] = pandas_df["provider_id"].isnull().sum() == 0
    
    return results

In [0]:
# Run validation and check results
print("=== DATA QUALITY VALIDATION ===")
print("=" * 50)

results = validate_claims(bronze_claims)

all_passed = True

for rule, passed in results.items():
    status = "✅ PASSED" if passed else "❌ FAILED"
    print(f"{rule}: {status}")
    if not passed:
        all_passed = False

print("=" * 50)

if all_passed:
    print("ALL CHECKS PASSED — Safe to process into Silver")
else:
    print("DATA QUALITY FAILED — Do not process into Silver")
    raise Exception("Data quality checks failed")

=== DATA QUALITY VALIDATION ===
claim_id not null: ✅ PASSED
claim_amount positive: ✅ PASSED
claim_status valid values: ✅ PASSED
patient_id not null: ✅ PASSED
provider_id not null: ✅ PASSED
ALL CHECKS PASSED — Safe to process into Silver


In [0]:
# Create bad data to test validation
bad_data = [
    ("CLM009", "PAT109", "PRV01", "APPROVED",  1500.00, "2024-01-17 10:00:00"),
    ("CLM010", None,     "PRV02", "PENDING",    800.00,  "2024-01-17 11:00:00"),  # null patient_id
    ("CLM011", "PAT111", "PRV03", "UNKNOWN",    2200.00, "2024-01-17 12:00:00"),  # invalid status
    ("CLM012", "PAT112", "PRV01", "APPROVED",  -500.00,  "2024-01-17 13:00:00"),  # negative amount
]

bad_df = spark.createDataFrame(
    bad_data,
    ["claim_id", "patient_id", "provider_id", "claim_status", "claim_amount", "updated_at"]
)

print("Bad data created:")
bad_df.show()

Bad data created:
+--------+----------+-----------+------------+------------+-------------------+
|claim_id|patient_id|provider_id|claim_status|claim_amount|         updated_at|
+--------+----------+-----------+------------+------------+-------------------+
|  CLM009|    PAT109|      PRV01|    APPROVED|      1500.0|2024-01-17 10:00:00|
|  CLM010|      NULL|      PRV02|     PENDING|       800.0|2024-01-17 11:00:00|
|  CLM011|    PAT111|      PRV03|     UNKNOWN|      2200.0|2024-01-17 12:00:00|
|  CLM012|    PAT112|      PRV01|    APPROVED|      -500.0|2024-01-17 13:00:00|
+--------+----------+-----------+------------+------------+-------------------+



In [0]:
# Run validation on bad data
print("=== TESTING BAD DATA ===")
print("=" * 50)

results = validate_claims(bad_df)

all_passed = True

for rule, passed in results.items():
    status = "✅ PASSED" if passed else "❌ FAILED"
    print(f"{rule}: {status}")
    if not passed:
        all_passed = False

print("=" * 50)

if all_passed:
    print("ALL CHECKS PASSED — Safe to process into Silver")
else:
    print("DATA QUALITY FAILED — Do not process into Silver")

=== TESTING BAD DATA ===
claim_id not null: ✅ PASSED
claim_amount positive: ❌ FAILED
claim_status valid values: ❌ FAILED
patient_id not null: ❌ FAILED
provider_id not null: ✅ PASSED
DATA QUALITY FAILED — Do not process into Silver


In [0]:
# Complete pipeline with data quality
def run_pipeline_with_quality(pipeline_name, source_table,
                               target_table, merge_key, watermark_col):
    
    print(f"Starting pipeline: {pipeline_name}")
    print("=" * 50)
    
    # Step 1 - Read watermark
    watermark_df = spark.sql("""
        SELECT last_watermark
        FROM pipeline_control.watermarks
        WHERE pipeline_name = '{}'
    """.format(pipeline_name))
    last_watermark = watermark_df.collect()[0][0]
    print(f"Last watermark: {last_watermark}")
    
    # Step 2 - Read Bronze incrementally
    bronze_df = spark.sql("""
        SELECT * FROM {}
        WHERE {} > '{}'
    """.format(source_table, watermark_col, last_watermark))
    print(f"Found {bronze_df.count()} new records")
    
    # Step 3 - Check if data exists
    if bronze_df.count() == 0:
        print("No new records — skipping")
        print("=" * 50)
        return
    
    # Step 4 - Run Great Expectations validation
    if "claims" in source_table:
        print("Running data quality checks...")
        results = validate_claims(bronze_df)
        all_passed = True
        for rule, passed in results.items():
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"  {rule}: {status}")
            if not passed:
                all_passed = False
        
        if not all_passed:
            raise Exception(f"Data quality failed for {pipeline_name}")
        
        print("All quality checks passed ✅")
    
    print("Pipeline complete:", pipeline_name)
    print("=" * 50)